In [11]:
#Import necessary libraries (e.g., pandas, numpy)
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import AutoModelForSeq2SeqLM

In [12]:
#qwen
def run_huggingface_qwen():
    """End-to-end text generation example with a Hugging Face causal LM."""

    # Step 1: Choose a small instruct model from the Qwen2.5 family.
    model_name = "Qwen/Qwen2.5-0.5B-Instruct"

    # Step 2: Load tokenizer.
    # The tokenizer converts text into token IDs that the model can process.
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Step 3: Load model weights.
    # device_map="auto" lets Transformers place the model automatically.
    # torch_dtype="auto" chooses a sensible precision if supported.
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype="auto",
        device_map="auto",
    )

    # Step 4: Build a chat-style prompt.
    # apply_chat_template formats the conversation in the model's expected style.
    messages = [
        {"role": "system", "content": "You are a helpful NLP tutor."},
        {"role": "user", "content": "Explain the difference between an RNN and a Transformer in 4 lines."},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # Step 5: Convert prompt text into tensors.
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Step 6: Generate continuation tokens.
    # max_new_tokens controls output length.
    # do_sample=True enables stochastic sampling for more natural text.
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

    # Step 7: Remove the prompt tokens so we decode only the new answer.
    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)

    print("=== HUGGING FACE: QWEN2.5 TEXT GENERATION ===")
    print(response)

    return tokenizer, model, response

tokenizer_qwen, model_qwen, response_qwen = run_huggingface_qwen()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

=== HUGGING FACE: QWEN2.5 TEXT GENERATION ===
An RNN (Recurrent Neural Network) is a type of neural network designed to process sequential data, such as sequences of words or images. It uses memory cells to store information about the state of the sequence at any given time. An RNN has two main layers: an input layer that takes the current sequence inputs, followed by an output layer that outputs the final sequence predictions.

A Transformer (also known as self-attentional transformer) is a type of recurrent neural network designed for tasks like language modeling. It introduces self-attention mechanisms that allow the network to focus on specific parts of the


In [13]:
def run_huggingface_nllb():
    
    model_name = "facebook/nllb-200-distilled-600M"
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    
    return tokenizer, model


# Load once
tokenizer_nllb, model_nllb = run_huggingface_nllb()


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

In [14]:
def translate_nllb(text, target_lang):
    """Translate English text into French or Spanish using NLLB"""
    
    # Step 1: Set source language
    tokenizer_nllb.src_lang = "eng_Latn"
    
    # Step 2: Tokenize input
    inputs = tokenizer_nllb(text, return_tensors="pt")
    
    # Step 3: Select target language
    if target_lang == "French":
        target_code = "fra_Latn"
    elif target_lang == "Spanish":
        target_code = "spa_Latn"
        
    # Step 4: Generate translation
    translated_tokens = model_nllb.generate(
        **inputs,
        forced_bos_token_id=tokenizer_nllb.convert_tokens_to_ids(target_code),
        max_new_tokens=120,
    )
    
    # Step 5: Decode
    translation = tokenizer_nllb.batch_decode(
        translated_tokens, skip_special_tokens=True
    )[0]
    
    return translation

In [15]:
# main function
def multilingual_chat(text):
    print(f'Your question in English: "{text}"\n')
    
    # 1. QWEN GENERATION
    messages = [
    {"role": "system", "content": "You are a helpful NLP tutor."},
    {"role": "user", "content": text},
    ]
    
    prompt = tokenizer_qwen.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    
    inputs = tokenizer_qwen(prompt, return_tensors="pt").to(model_qwen.device)
    
    outputs = model_qwen.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )
    
    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    answer_en = tokenizer_qwen.decode(generated_ids, skip_special_tokens=True)
    
    print("Answer in English:", answer_en, "\n")

    # 2. NLLB TRANSLATION
    answer_fr = translate_nllb(answer_en, "French")
    answer_es = translate_nllb(answer_en, "Spanish")
    
    print("Answer in French:", answer_fr, "\n")
    print("Answer in Spanish:", answer_es)
    



In [ ]:
#test

#print(tokenizer_qwen)
#print(model_qwen)
multilingual_chat("Explain what a neural network is")

multilingual_chat("Please explain vanishing gradient and exploding gradient")

multilingual_chat("What is the difference between an RNN and an LSTM?")

multilingual_chat("Why are Transformers better at handling long-range dependencies?")

Your question in English: "Explain what a neural network is"

Answer in English: A neural network is an artificial intelligence model that mimics the structure and function of biological neurons in the human brain. It consists of multiple layers of interconnected nodes, or neurons, that process information by passing it along to subsequent nodes based on their inputs.

Neural networks use various activation functions to determine which output neuron should receive the next input, rather than directly applying weights to each input. This allows them to learn from data and make predictions about new, unseen inputs without being explicitly programmed with formulas.

Key components include:

1. **Input Layer**: A layer of neurons that receives the input data.
2. **Hidden Layers**: Intermediate layers that pass the incoming inputs through a series of non-linear transformations.
3. **Output Layer**: A final layer that 



[transformers] Both `max_new_tokens` (=120) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer in French: Un réseau neural est un modèle d'intelligence artificielle qui imite la structure et la fonction des neurones biologiques dans le cerveau humain. Il se compose de plusieurs couches de nœuds interconnectés, ou neurones, qui traitent l'information en la transmettant aux nœuds suivants en fonction de leurs entrées. Les réseaux neuraux utilisent diverses fonctions d'activation pour déterminer quelle neurone de sortie doit recevoir l'entrée suivante, plutôt que d'appli 

Answer in Spanish: Una red neuronal es un modelo de inteligencia artificial que imita la estructura y función de las neuronas biológicas en el cerebro humano. Consiste en múltiples capas de nodos interconectados, o neuronas, que procesan la información pasando a través de los nodos posteriores basados en sus entradas. Las redes neuronales utilizan varias funciones de activación para determinar qué neurona de salida debe recibir la próxima entrada, en lugar de aplicar pesos directamente a cada entrada. Esto

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]